
Aprovechandonos del ejemplo de uno, resulta que el equipo de marketing necesita saber como van las ventas para poder organizar las promociones de forma más agresiva. Sin embargo se ha hablado con los directores de las tiendas y se ha decidido que no las tienes del norte y del sur NO participarán en el programa de marketing. ¿Como solucionamos esto?

In [0]:
%sql

/* Para solucionar esto, podríamos crear una vista, pero es mucho más sencillo añadir un row_filter a la tabla, y de paso repasamos los grants de los equipos */ 

GRANT USE CATALOG ON CATALOG workspace TO `sales`;
GRANT USE SCHEMA ON SCHEMA workspace.gold TO `sales`;
GRANT SELECT ON TABLE workspace.gold.private_sales TO `sales`;
GRANT USE CATALOG ON CATALOG workspace TO `marketing`;
GRANT USE SCHEMA ON SCHEMA workspace.gold TO `marketing`;
GRANT SELECT ON TABLE workspace.gold.private_sales TO `marketing`;

CREATE FUNCTION IF NOT EXISTS stores_filter(store STRING)
RETURN IF(IS_ACCOUNT_GROUP_MEMBER('sales'), true, store NOT IN ('Store_North', 'Store_South'));

ALTER TABLE workspace.gold.private_sales SET ROW FILTER stores_filter ON (store);



Después de discutir, el equipo de marketing no está contento. Insisten en que esta aproximación no tiene mucho sentido, ya que no pueden comparar todas las tiendas para hacer un baseline, ¿y si hacen ofertas sobre tiendas que ya están vendiendo mucho más por encima de la media? 

In [0]:
%sql
 -- Eliminamos el filtro y hacemos un column masking
ALTER TABLE workspace.gold.private_sales DROP ROW FILTER;



In [0]:
%sql
CREATE OR REPLACE FUNCTION store_mask(store STRING)
  RETURN CASE WHEN IS_ACCOUNT_GROUP_MEMBER('marketing') THEN
    CASE 
      WHEN lower(store) IN ('store_north') THEN 'STORE_01'
      WHEN lower(store) IN ('store_south') THEN 'STORE_02'
      ELSE store
    END
  ELSE store
END;
ALTER TABLE workspace.gold.private_sales ALTER COLUMN store SET MASK store_mask;



In [0]:
%sql
-- Como soy miembro del equipo de marketing... No veo las tiendas. 
select * from  workspace.gold.private_sales